In [1]:
%pip install llama-parse
%pip install sentence-transformers
%pip install numpy
%pip install langchain
%pip install python-dotenv
%pip install groq
%pip install pinecone[grpc]
%pip install rank-bm25 nltk numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

os.environ["PINECONE_API_KEY"] = "pcsk_5opcAf_JtDsfsvUmdeQsgpRK78iLqDzfFEYAVdwUr1C1xqfmWYYxdpwdEhSDPenXntSZUv"
os.environ["GROQ_API_KEY"] = "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"

print("✅ Environment variables set")

✅ Environment variables set


In [8]:
import re
from typing import List, Dict
from datetime import datetime

class DocumentChunker:
    def __init__(self, max_tokens: int = 450):
        self.metadata = {
            "last_processed": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

        self.max_tokens = max_tokens
        # Rata-rata 1 token ≈ 4 karakter untuk bahasa Indonesia/Inggris
        self.approx_chars_per_token = 4
        self.max_chars = max_tokens * self.approx_chars_per_token

    def _clean_text(self, text: str) -> str:
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def _remove_heading_markers(self, text: str) -> str:
        text = re.sub(r'^#{1,6}\s+', '', text)
        return text.strip()

    def _preprocess_content(self, content: str) -> str:
        content = re.sub(r'\n{2,}', '\n', content)
        content = re.sub(r'\|\s*\n\s*\|', '|\\n|', content)
        content = re.sub(r'\|\s*---\s*\|', '|---|', content)
        return content
    
    def _extract_source(self, content: str) -> str:
        match = re.match(r'^#\s+(.+?)(?:\n|$)', content)
        if match:
            return match.group(1).strip()
        return "Unknown Source"

    def _remove_h1_header(self, content: str) -> str:
        """
        Hapus baris H1 dari konten jika langsung diikuti H2.
        Ini mencegah H1 menjadi chunk terpisah.
        """
        # Pattern: H1 di awal, diikuti baris kosong (opsional), lalu H2
        pattern = r'^#\s+[^\n]+\n+(?=##\s)'
        return re.sub(pattern, '', content)

    def _merge_consecutive_headings(self, content: str) -> str:
        pattern = r'(##\s+[^\n]+)\n(###\s+[^\n]+)'
        
        def merge_headings(match):
            h2 = match.group(1)
            h3_text = match.group(2).lstrip('#').strip()
            return f"{h2} | {h3_text}"
        
        return re.sub(pattern, merge_headings, content)

    def _extract_chunk_title(self, text: str, section_title: str) -> str:
        """
        Ekstrak title dari chunk.
        - Jika chunk dimulai dengan heading, gunakan heading tersebut
        - Jika tidak, gunakan parent_title
        """
        # Cek apakah dimulai dengan heading (##, ###, dll)
        match = re.match(r'^(#{2,6})\s+([^\n]+)', text)
        if match:
            return self._remove_heading_markers(match.group(0))
        return section_title

    def _split_long_text(self, text: str) -> List[str]:
        if len(text) <= self.max_chars:
            return [text]
        
        parts = []
        current_text = text
        
        while len(current_text) > self.max_chars:
            cut_point = self.max_chars
            
            for delimiter in ['. ', '! ', '? ', '\n']:
                last_delimiter = current_text[:self.max_chars].rfind(delimiter)
                if last_delimiter > self.max_chars * 0.5:
                    cut_point = last_delimiter + len(delimiter)
                    break
            else:
                last_space = current_text[:self.max_chars].rfind(' ')
                if last_space > self.max_chars * 0.5:
                    cut_point = last_space + 1
            
            parts.append(current_text[:cut_point].strip())
            current_text = current_text[cut_point:].strip()
        
        if current_text:
            parts.append(current_text)
        
        return parts

    def process_documentation(self, input_path: str) -> List[Dict]:
        with open(input_path, 'r', encoding='utf-8') as file:
            content = file.read()

        from pathlib import Path
        file_prefix = Path(input_path).stem

        content = self._preprocess_content(content)
        document_source = self._extract_source(content)
        content = self._remove_h1_header(content)
        content = self._merge_consecutive_headings(content)
        
        chunks = []
        sections = re.split(r'\n\s*##\s+(?=[A-Za-z0-9])', content)

        for section_idx, section in enumerate(sections):
            if not section.strip():
                continue

            title_match = re.match(r'^([^\n]+)', section)
            raw_title = title_match.group(1) if title_match else "Untitled Section"
            section_title = self._remove_heading_markers(raw_title)

            subsections = []
            current_chunk = []

            for line in section.split('\n'):
                if line.startswith('|'):
                    current_chunk.append(line)
                elif line.startswith('###'):
                    if current_chunk:
                        subsections.append('\n'.join(current_chunk))
                        current_chunk = []
                    current_chunk.append(line)
                elif line.startswith('##'):
                    if current_chunk:
                        subsections.append('\n'.join(current_chunk))
                        current_chunk = []
                    current_chunk.append(line)
                else:
                    current_chunk.append(line)

            if current_chunk:
                subsections.append('\n'.join(current_chunk))

            chunk_counter = 0

            for subsection in subsections:
                chunk_title = self._extract_chunk_title(subsection, section_title)
                
                cleaned_content = self._clean_text(subsection)
                if not cleaned_content:
                    continue
                
                cleaned_content = self._remove_heading_markers(cleaned_content)
                # Split jika melebihi batas token
                content_parts = self._split_long_text(cleaned_content)
                
                for part_content in content_parts:
                    chunk = {
                        "id": f"{file_prefix}_section_{section_idx}_chunk_{chunk_counter}",
                        "content": part_content,
                        "metadata": {
                            "source": document_source,
                            "section_title": chunk_title,
                            "sequence": chunk_counter
                        }
                    }
                    chunks.append(chunk)
                    chunk_counter += 1  # Increment untuk setiap chunk (termasuk split)
        print(f"Total chunks created: {len(chunks)}")
        return chunks
print("✅ DocumentChunker defined")

✅ DocumentChunker defined


In [9]:
from pathlib import Path

# Path ke folder data
DATA_DIR = Path("e:/Rag Chatbor Learning/try-good-rag/data")

# Inisialisasi chunker
chunker = DocumentChunker()

# Proses semua file markdown
all_chunks = []

for md_file in DATA_DIR.glob("*.md"):
    print(f"\nProcessing: {md_file.name}")
    chunks = chunker.process_documentation(str(md_file))
    all_chunks.extend(chunks)

print(f"\n{'='*50}")
print(f"Total chunks dari semua dokumen: {len(all_chunks)}")

# Preview
print("\nExample chunk:")
if all_chunks:
    print(all_chunks[0])


Processing: bab2kurikulumops.md
Total chunks created: 6

Processing: bab3kurikulumops.md
Total chunks created: 29

Processing: bab4kurikulumops.md
Total chunks created: 2

Processing: profil.md
Total chunks created: 5

Total chunks dari semua dokumen: 42

Example chunk:
{'id': 'bab2kurikulumops_section_0_chunk_0', 'content': "Visi Sekolah Terwujudnya generasi masa depan yang taqwa, cerdas dan mandiri Indikator: 1. Indikator Taqwa Menumbuhkan karakter spiritual keagamaan peserta didik dengan: - Tumbuh kesadaran menjalankan sholat wajib dan sunnah, serta ibadah sehari-hari lainnya. - Senang dan terampil membaca Al-Qur'an serta mampu menghafal minimal 1 sampai 2 juz (juz 29 dan 30). - Memiliki pemahaman aqidah dan syakhsiah islamiyah (kepribadian islam) yang benar. 2. Indikator Cerdas Menumbuhkan karakter pembelajar dan terampil peserta didik dengan: - Tumbuh minat belajar yang tinggi. - Gemar membaca dan menulis. - Berpikir logis dan analitis. - Memiliki prestasi akademik yang benar. 3.

In [10]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List, Dict
import time
from tqdm import tqdm
import os

class DocumentEmbedder:
    def __init__(self, model_name: str = 'intfloat/multilingual-e5-base'):
        print(f"\nInitializing DocumentEmbedder...")
        print(f"├── Model: {model_name}")

        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()

        api_key = os.getenv('PINECONE_API_KEY')
        if not api_key:
            raise ValueError("PINECONE_API_KEY not found")

        self.pc = Pinecone(api_key=api_key)
        self.index_name = "school"  # ✅ Konsisten dengan HybridSearcher
        self._initialize_index()

    def _initialize_index(self):
        try:
            existing_indexes = [idx.name for idx in self.pc.list_indexes()]

            if self.index_name not in existing_indexes:
                print(f"├── Creating index: {self.index_name}")
                self.pc.create_index(
                    name=self.index_name,
                    dimension=self.embedding_dim,
                    metric="cosine",
                    spec=ServerlessSpec(
                        cloud="aws",
                        region="us-east-1"
                    )
                )
                while not self.pc.describe_index(self.index_name).status['ready']:
                    time.sleep(1)

            self.index = self.pc.Index(self.index_name)
            print(f"├── Connected to index: {self.index_name}")

        except Exception as e:
            raise Exception(f"Failed to initialize index: {str(e)}")

    def generate_embeddings(self, chunks: List[Dict[str, str]]) -> np.ndarray:
        texts = [f"passage: {chunk['content']}" for chunk in chunks]
        print(f"\nGenerating embeddings:")
        print(f"├── Total chunks: {len(texts)}")
        print(f"└── Model: {self.model_name}")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True,
            batch_size=32,
            normalize_embeddings=True
        )
        return embeddings

    def process_chunks(self, chunks: List[Dict[str, str]], batch_size: int = 100):
        try:
            embeddings = self.generate_embeddings(chunks)

            print("\nPreparing vectors for Pinecone...")
            vectors = [
                {
                    'id': chunk['id'],
                    'values': embedding.tolist(),
                    'metadata': {
                        'content': chunk['content'],
                        'source': chunk['metadata']['source'],
                        'section_title': chunk['metadata']['section_title'],
                        'sequence': chunk['metadata']['sequence']
                    }
                }
                for chunk, embedding in zip(chunks, embeddings)
            ]

            print("\nUploading to Pinecone...")
            for i in tqdm(range(0, len(vectors), batch_size)):
                batch = vectors[i:i + batch_size]
                self.index.upsert(vectors=batch)

            print(f"\n✅ Processing complete:")
            print(f"├── Total chunks: {len(chunks)}")
            print(f"├── Model: {self.model_name}")
            print(f"├── Dimension: {self.embedding_dim}")
            print(f"└── Index: {self.index_name}")

            return True

        except Exception as e:
            print(f"\n❌ Error processing chunks: {str(e)}")
            raise

# Upload chunks ke Pinecone
embedder = DocumentEmbedder()
embedder.process_chunks(all_chunks)  # ✅ Menggunakan all_chunks dari Cell 4


Initializing DocumentEmbedder...
├── Model: intfloat/multilingual-e5-base
├── Creating index: school
├── Connected to index: school

Generating embeddings:
├── Total chunks: 42
└── Model: intfloat/multilingual-e5-base


Batches: 100%|██████████| 2/2 [00:29<00:00, 14.84s/it]



Preparing vectors for Pinecone...

Uploading to Pinecone...


100%|██████████| 1/1 [00:06<00:00,  6.64s/it]


✅ Processing complete:
├── Total chunks: 42
├── Model: intfloat/multilingual-e5-base
├── Dimension: 768
└── Index: school


True

In [7]:
from sentence_transformers import SentenceTransformer
from typing import List, Dict
from pinecone.grpc import PineconeGRPC as Pinecone
import os
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
import numpy as np

class HybridSearcher:
    def __init__(self, model_name: str = 'intfloat/multilingual-e5-base'):
        self.model = SentenceTransformer(model_name)


        load_dotenv()
        api_key = os.getenv('PINECONE_API_KEY')
        self.pc = Pinecone(api_key=api_key)
        self.index = self.pc.Index("school")

        # Initialize BM25 parameters
        self.similarity_threshold = 0.3
        self.alpha = 0.7  # Weight for vector search scores
        self.cached_docs = []
        self.bm25 = None

        print(f"Hybrid Searcher initialized with index: school")

    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization function"""
        # Convert to lowercase and split on whitespace and punctuation
        text = text.lower()
        # Replace common punctuation with spaces
        for char in ',.!?;:()[]{}""''':
            text = text.replace(char, ' ')
        # Split on whitespace and filter out empty strings
        return [token for token in text.split() if token]

    def _initialize_bm25(self, documents: List[str]):
        """Initialize BM25 with tokenized documents"""
        tokenized_docs = [self._tokenize(doc) for doc in documents]
        self.bm25 = BM25Okapi(tokenized_docs)
        self.cached_docs = documents

    def search(self, query: str, k: int = 3, hybrid_weight: float = 0.7) -> List[Dict]:
        try:
            # Generate query embedding for vector search
            query_with_prefix = f"query: {query}"
            query_embedding = self.model.encode(query_with_prefix)

            # Perform vector search
            vector_results = self.index.query(
                vector=query_embedding.tolist(),
                top_k=k * 2,
                include_metadata=True
            )

            # Extract documents and initialize BM25 if needed
            documents = [match.metadata['content'] for match in vector_results.matches]
            if documents != self.cached_docs:
                self._initialize_bm25(documents)

            # Get BM25 scores
            tokenized_query = self._tokenize(query)
            bm25_scores = self.bm25.get_scores(tokenized_query)

            # Normalize scores
            vector_scores = np.array([match.score for match in vector_results.matches])
            normalized_bm25_scores = bm25_scores / (np.max(bm25_scores) + 1e-10)
            normalized_vector_scores = vector_scores / (np.max(vector_scores) + 1e-10)

            # Combine scores
            hybrid_scores = (hybrid_weight * normalized_vector_scores +
                           (1 - hybrid_weight) * normalized_bm25_scores)

            # Create results with combined scores
            results = []
            for idx, match in enumerate(vector_results.matches):
                if hybrid_scores[idx] >= self.similarity_threshold:
                    result = {
                        'content': match.metadata['content'],
                        'source': match.metadata['source'],  
                        'sequence': match.metadata['sequence'],
                        'vector_score': float(vector_scores[idx]),
                        'bm25_score': float(bm25_scores[idx]),
                        'hybrid_score': float(hybrid_scores[idx]),
                        'section_title': match.metadata['section_title'],
                        'metadata': match.metadata
                    }
                    results.append(result)

            # Return top k unique results
            return self._deduplicate_results(results)[:k]

        except Exception as e:
            print(f"Error during search: {str(e)}")
            return []

    def _deduplicate_results(self, results: List[Dict]) -> List[Dict]:
        seen_content = set()
        unique_results = []

        for result in sorted(results, key=lambda x: x['hybrid_score'], reverse=True):
            content_hash = hash(result['content'])
            if content_hash not in seen_content:
                seen_content.add(content_hash)
                unique_results.append(result)

        return unique_results



searcher = HybridSearcher()
results = searcher.search("apa mapel kelas 1", k=3, hybrid_weight=0.7)


for idx, result in enumerate(results, 1):
    print(f"\nResult {idx}")
    print(f"Source: {result['source']}")
    print(f"Hybrid Score: {result['hybrid_score']:.3f}")
    print(f"Vector Score: {result['vector_score']:.3f}")
    print(f"BM25 Score: {result['bm25_score']:.3f}")
    print(f"Section: {result['section_title']}")
    print(f"Sequence: {result['sequence']}")
    print(f"Content: {result['content'][:200]}...")



Hybrid Searcher initialized with index: school

Result 1
Source: Kurikulum Operasional SD Integral Luqman Al Hakim Situbondo BAB III PENGORGANISASIAN PEMBELAJARAN DAN RENCANA PEMBELAJARAN
Hybrid Score: 0.996
Vector Score: 0.802
BM25 Score: 0.219
Section: 5. Beban Belajar
Sequence: 1.0
Content: ### Tabel 5.1 Pengaturan waktu belajar untuk kelas I atau 1 adalah sebagai berikut: | No | Mata Pelajaran | Banyak JP Per Minggu | Kegiatan Reguler Per Tahun | Proyek Profil Pelajar Pancasila | Total ...

Result 2
Source: Profil SD Integral Luqman Al Hakim Situbondo Tahun Ajaran 2025/2026
Hybrid Score: 0.988
Vector Score: 0.788
BM25 Score: 0.222
Section: Data Guru/Karyawan, Murid dan Rombongan Belajar Kelas
Sequence: 0.0
Content: Data Guru/Karyawan, Murid dan Rombongan Belajar Kelas - Jumlah Guru/Karyawan : 39 orang - Jumlah Murid : 362 anak - Jumlah Rombel Kelas : 19 kelas...

Result 3
Source: Kurikulum Operasional SD Integral Luqman Al Hakim Situbondo BAB III PENGORGANISASIAN PEMBELAJARAN DAN R

In [8]:
from groq import Groq
from typing import List, Dict
import os
from dotenv import load_dotenv

class GroqService:
    def __init__(self):
        load_dotenv()
        # Ganti GOOGLE_API_KEY dengan GROQ_API_KEY di file .env
        self.client = Groq(api_key=os.getenv('GROQ_API_KEY'))
        self.model = "llama-3.3-70b-versatile"

    def _prepare_prompt(self, question: str, search_results: List[Dict]) -> str:
        context_text = "\n".join([result['content'] for result in search_results])

        system_prompt = """Anda adalah asisten informasi resmi untuk Sekolah Dasar Integral Luqman Al Hakim Situbondo. 
        Tugas utama Anda adalah memberikan informasi yang akurat dan membantu berdasarkan dokumen resmi sekolah.
        ATURAN UTAMA:
        1. HANYA berikan informasi yang TERSEDIA dalam konteks dokumen yang diberikan
        2. JANGAN pernah mengarang, mengasumsikan, atau membuat informasi yang tidak ada dalam dokumen
        3. Jika informasi TIDAK DITEMUKAN dalam konteks, jawab dengan jujur:
        "Mohon maaf, informasi mengenai [topik] tidak tersedia dalam dokumen yang saya miliki. 
        Silakan hubungi pihak sekolah secara langsung untuk informasi lebih lanjut."
        4. Gunakan Bahasa Indonesia yang baik, sopan, dan mudah dipahami oleh semua kalangan
        5. Jika pertanyaan ambigu atau kurang jelas, minta klarifikasi dengan sopan
        PANDUAN MENJAWAB:
        - Berikan jawaban yang ringkas namun lengkap
        - Gunakan format yang rapi dan mudah dibaca (poin-poin jika perlu)
        - Sebutkan sumber bagian dokumen jika relevan (misalnya: "Berdasarkan Bab 2 Kurikulum...")
        - Untuk data numerik (tahun, jumlah, biaya, dll), pastikan menyebutkan dengan tepat sesuai dokumen
        - Jika ada beberapa informasi terkait, rangkum dengan terstruktur
        - Gunakan nada ramah dan profesional layaknya staf administrasi sekolah
        TOPIK YANG MUNGKIN DITANYAKAN:
        - Profil sekolah (visi, misi, sejarah, alamat, kontak)
        - Struktur organisasi (kepala sekolah, guru, staf)
        - Kurikulum dan program pembelajaran
        - Kegiatan ekstrakurikuler
        - Tata tertib dan peraturan sekolah
        - Informasi pendaftaran dan penerimaan siswa baru
        - Fasilitas sekolah
        - Jadwal dan kalender akademik
        - Dan informasi sekolah lainnya"""
        
        user_message = f"""Konteks dari Dokumen Sekolah:
        ---
        {context_text}
        ---
        Pertanyaan: {question}
        Berikan jawaban berdasarkan konteks dokumen di atas. Ingat, jangan mengarang informasi yang tidak ada dalam dokumen."""
        
        return system_prompt, user_message

    def _format_response(self, text: str) -> str:
        """Format respons untuk tampilan yang lebih baik"""
        lines = text.split('\n')
        formatted_lines = []
        
        for line in lines:
            # Format heading
            if line.strip().startswith('#'):
                formatted_lines.append(f"\n{line}\n")
            # Format list items
            elif line.strip().startswith(('-', '*', '•')):
                formatted_lines.append(line)
            # Format numbered list
            elif line.strip() and line.strip()[0].isdigit() and '.' in line[:3]:
                formatted_lines.append(line)
            else:
                formatted_lines.append(line)
        return '\n'.join(formatted_lines)

    def get_answer(self, question: str, search_results: List[Dict]) -> str:
        try:
            if not search_results:
                return ("Mohon maaf, saya tidak menemukan informasi yang relevan dengan pertanyaan Anda "
                       "dalam dokumen sekolah. Silakan coba dengan kata kunci lain atau hubungi "
                       "pihak sekolah secara langsung.")
                       
            system_prompt, user_message = self._prepare_prompt(question, search_results)

            completion = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": user_message
                    }
                ],
                temperature=0.3,
                max_tokens=1024,
                top_p=0.9,
                stop=None,
                stream=False
            )
            
            # Cara mengambil respons dari Groq
            response_text = completion.choices[0].message.content
            return self._format_response(response_text.strip())

        except Exception as e:
            print(f"Groq API error: {str(e)}")
            return ("Mohon maaf, terjadi kendala teknis dalam memproses pertanyaan Anda. "
                   "Silakan coba beberapa saat lagi.")

In [ ]:
question = "siapa ketua yayasan dan kepala sekolah"
searcher = HybridSearcher()

results = searcher.search(question, k=8, hybrid_weight=0.7)
llm_service = GroqService()
answer = llm_service.get_answer(question, search_results=results)

print(answer)

Hybrid Searcher initialized with index: school
Berdasarkan konteks dokumen yang disediakan, informasi mengenai ketua yayasan dan kepala sekolah adalah sebagai berikut:

1. Ketua Yayasan: Mustaji, S.Pd
2. Kepala Sekolah: Imam Romli, M.Pd

Informasi ini ditemukan pada bagian "Identitas Sekolah" dalam dokumen.
